<a href="https://colab.research.google.com/github/Supriyagattu/NLP/blob/main/2230_nlp_A12_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 28.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import Word2Vec

In [ ]:
sample_texts = [
    "This movie was fantastic, I loved every moment of it!",
    "A terrible waste of time, absolutely boring and poorly acted.",
    "The plot was engaging, and the characters were well-developed.",
    "Could not finish it, the storyline was weak and the pacing was off.",
    "Highly recommend, a truly captivating cinematic experience.",
    "I expected more, it was just an average film with predictable twists."
]
labels = [1,1,0,0,1,0]


In [ ]:
tokenizer = Tokenizer()

# Learn vocabulary
tokenizer.fit_on_texts(sample_texts)

# Convert sentences into sequences
sequences = tokenizer.texts_to_sequences(sample_texts)

# Vocabulary dictionary
word_index = tokenizer.word_index

print(word_index)


{'was': 1, 'the': 2, 'it': 3, 'and': 4, 'i': 5, 'of': 6, 'a': 7, 'this': 8, 'movie': 9, 'fantastic': 10, 'loved': 11, 'every': 12, 'moment': 13, 'terrible': 14, 'waste': 15, 'time': 16, 'absolutely': 17, 'boring': 18, 'poorly': 19, 'acted': 20, 'plot': 21, 'engaging': 22, 'characters': 23, 'were': 24, 'well': 25, 'developed': 26, 'could': 27, 'not': 28, 'finish': 29, 'storyline': 30, 'weak': 31, 'pacing': 32, 'off': 33, 'highly': 34, 'recommend': 35, 'truly': 36, 'captivating': 37, 'cinematic': 38, 'experience': 39, 'expected': 40, 'more': 41, 'just': 42, 'an': 43, 'average': 44, 'film': 45, 'with': 46, 'predictable': 47, 'twists': 48}


In [ ]:
print(sequences)

[[8, 9, 1, 10, 5, 11, 12, 13, 6, 3], [7, 14, 15, 6, 16, 17, 18, 4, 19, 20], [2, 21, 1, 22, 4, 2, 23, 24, 25, 26], [27, 28, 29, 3, 2, 30, 1, 31, 4, 2, 32, 1, 33], [34, 35, 7, 36, 37, 38, 39], [5, 40, 41, 3, 1, 42, 43, 44, 45, 46, 47, 48]]


In [ ]:
max_length = 6

X = pad_sequences(sequences, maxlen=max_length)

y = np.array(labels)


In [ ]:
X

array([[ 5, 11, 12, 13,  6,  3],
       [16, 17, 18,  4, 19, 20],
       [ 4,  2, 23, 24, 25, 26],
       [31,  4,  2, 32,  1, 33],
       [35,  7, 36, 37, 38, 39],
       [43, 44, 45, 46, 47, 48]], dtype=int32)

In [ ]:
sentences = [text.split() for text in sample_texts]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences,
    vector_size=50,
    window=3,
    min_count=1
)

In [ ]:
vocab_size = len(word_index) + 1 # +1 for the 0th index (padding)
embedding_dim = 50 # Must match vector_size used in Word2Vec

embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
    else:
        # Handle words not in Word2Vec vocabulary if any (e.g., set to zero vector or random vector)
        pass

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print("First 5 rows of embedding matrix:\n", embedding_matrix[:5])


Embedding matrix shape: (49, 50)
First 5 rows of embedding matrix:
 [[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-1.03726867e-03  4.77704540e-04  1.02141341e-02  1.80738308e-02
  -1.86071433e-02 -1.42610455e-02  1.29

In [ ]:
input_layer = Input(shape=(max_length,))


In [ ]:
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],  # Word2Vec weights
    input_length=max_length,
    trainable=False               # keep embeddings fixed
)(input_layer)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
conv1 = Conv1D(filters=100, kernel_size=3, activation='relu')(embedding_layer)

conv2 = Conv1D(filters=100, kernel_size=4, activation='relu')(embedding_layer)

conv3 = Conv1D(filters=100, kernel_size=5, activation='relu')(embedding_layer)



In [ ]:
pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)

In [ ]:
merged = Concatenate()([pool1, pool2, pool3])


In [ ]:
dense = Dense(10, activation='relu')(merged)


In [ ]:
output = Dense(1, activation='sigmoid')(dense)


In [ ]:
model = Model(inputs=input_layer, outputs=output)

In [ ]:
# Input layer
sequence_input = Input(shape=(max_length,), dtype='int32')

# Embedding layer (using pre-trained Word2Vec embeddings)
embedding_layer = Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    input_length=max_length,
    trainable=False # Set to False to keep embeddings static, True to fine-tune
)(sequence_input)

# Convolutional layers with different kernel sizes
conv_blocks = []
kernel_sizes = [3, 4, 5]
for kernel_size in kernel_sizes:
    conv = Conv1D(filters=128, kernel_size=kernel_size, activation='relu')(embedding_layer)
    pool = GlobalMaxPooling1D()(conv)
    conv_blocks.append(pool)

# Concatenate the outputs of different convolutional blocks
merged = Concatenate()(conv_blocks)

# Dense layers
dense = Dense(128, activation='relu')(merged)
output = Dense(1, activation='sigmoid')(dense) # Binary classification, so 1 output unit with sigmoid

# Define the model
model = Model(sequence_input, output)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 6, 50)     │      2,450 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 4, 128)    │     19,328 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 3, 128)    │     25,728 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 2, 128)    │     32,128 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_3[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_4[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_5[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 384)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     49,280 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │        129 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 129,043 (504.07 KB)

 Trainable params: 126,593 (494.50 KB)

 Non-trainable params: 2,450 (9.57 KB)

In [ ]:
history = model.fit(X, y, epochs=10, verbose=1, validation_split=0.2)

print("\nModel trained successfully!")

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 1.0000 - loss: 0.5622 - val_accuracy: 0.5000 - val_loss: 0.6945
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 1.0000 - loss: 0.5438 - val_accuracy: 0.5000 - val_loss: 0.6947
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 1.0000 - loss: 0.5241 - val_accuracy: 0.5000 - val_loss: 0.6948
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 1.0000 - loss: 0.5033 - val_accuracy: 0.5000 - val_loss: 0.6950
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 1.0000 - loss: 0.4813 - val_accuracy: 0.5000 - val_loss: 0.6952
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 1.0000 - loss: 0.4583 - val_accuracy: 0.5000 - val_loss: 0.6955
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 1.0000 - loss: 0.4344 - val_accuracy: 0.5000 - val_loss: 0.6961
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 1.0000 - loss: 0.4098 - val_accuracy: 0.5000 - val_loss: 0.69

In [ ]:
test_text = ["this movie is amazing"]

seq = tokenizer.texts_to_sequences(test_text)

pad = pad_sequences(seq, maxlen=max_length)

prediction = model.predict(pad)

print(prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
[[0.49985147]]
